# RAG vs Graph RAG vs Hybrid — Hands-on Walkthrough
### AAA Learning Session · evening hands-on

This notebook builds **three retrieval architectures from scratch** on the same small corpus, so you can *see* exactly where each one wins:

| Part | Build | What it teaches |
|------|-------|-----------------|
| **1** | Traditional (vector) RAG | Chunk → embed → FAISS → retrieve → generate |
| **2** | Graph RAG (from scratch) | LLM entity/relation extraction → knowledge graph → traversal |
| **3** | Hybrid router | Classify each query → route to the right engine |
| **4** | Compare & evaluate | Same questions through all three, side by side |

> **How to run:** `Runtime → Run all`, or run cells top-to-bottom. You only need an LLM API key (cell 0.2). Everything runs on Colab's free tier.

This is a *teaching* implementation — small, readable, and fully in one notebook. The final section maps each idea to the production libraries (Microsoft GraphRAG, LightRAG, Neo4j) you'd use in the real world.


## 0 · Setup

### 0.1 Install dependencies
Minimal stack: `openai` (LLM + embeddings), `faiss-cpu` (vector search), `networkx` (the graph).

In [ ]:
%pip install -q openai faiss-cpu networkx numpy tiktoken
print("✅ dependencies installed")

### 0.2 API key
Defaults to **OpenAI** (`gpt-4o-mini` is cheap and plenty for a demo). In Colab, add your key in the 🔑 *Secrets* panel as `OPENAI_API_KEY`, or paste it when prompted.

> Prefer another provider or fully local? See the final note — any chat + embedding endpoint works; only the two helpers in 0.3 change.

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY") or userdata.get("OPENAI_API_KEY")
except Exception:
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OPENAI_API_KEY: ")
assert os.environ.get("OPENAI_API_KEY"), "No API key set."
print("✅ API key loaded")

### 0.3 LLM + embedding helpers
Two tiny wrappers we reuse everywhere. Swapping providers later means editing **only these two functions**.

In [ ]:
from openai import OpenAI
import numpy as np

client = OpenAI()
CHAT_MODEL  = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

def llm(prompt, system="You are a precise, helpful assistant.", temperature=0.0):
    """Single-turn chat completion -> string."""
    resp = client.chat.completions.create(
        model=CHAT_MODEL, temperature=temperature,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()

def embed(texts):
    """List[str] -> (n, dim) float32, L2-normalized (cosine == inner product)."""
    if isinstance(texts, str): texts = [texts]
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    v = np.array([d.embedding for d in resp.data], dtype="float32")
    v /= (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)
    return v

print(llm("Reply with exactly: ready"))
print("embedding dim:", embed("hello").shape)

## 1 · A small corpus *designed* to expose the difference

A compact, fictional tech-ecosystem corpus, deliberately built so **facts are scattered across documents** and connected through entities (companies, people, products, acquisitions).

Notice no single paragraph answers *"How is Lena Ortiz connected to NimbusDB?"* — you must **hop**:
`Ortiz → founded Vexel → acquired by Cumulo → which builds NimbusDB`. That multi-hop chain is where naive RAG struggles and a graph shines.

In [ ]:
DOCS = {
 "doc1_orion": """Orion Labs is an AI infrastructure company founded in 2019 by Marcus Feld.
Orion Labs builds Pulse, a real-time vector database used for retrieval systems.
In 2023 Orion Labs partnered with Cumulo Cloud to offer Pulse as a managed service.""",

 "doc2_cumulo": """Cumulo Cloud is a cloud platform provider. Its flagship product is NimbusDB,
a managed graph database. In 2022 Cumulo Cloud acquired Vexel Analytics to strengthen
its analytics stack. Cumulo Cloud is led by CEO Priya Anand.""",

 "doc3_vexel": """Vexel Analytics was a startup specializing in knowledge-graph tooling.
Vexel Analytics was founded by Lena Ortiz. After the acquisition, Ortiz became
VP of Graph Products at Cumulo Cloud, where she now oversees NimbusDB.""",

 "doc4_pulse": """Pulse is optimized for low-latency similarity search and is commonly paired
with embedding models. Teams use Pulse for traditional RAG pipelines. Pulse integrates
with NimbusDB so applications can combine vector search with graph traversal.""",

 "doc5_market": """The retrieval market in 2024 splits into vector-first and graph-first camps.
Analysts note that hybrid systems — combining a vector store like Pulse with a graph store
like NimbusDB — are becoming the default for enterprise knowledge assistants.""",
}
for k, v in DOCS.items():
    print(f"— {k:14s} ({len(v.split())} words)")

---
## Part 1 · Traditional (vector) RAG

The classic pipeline. Four offline steps, two at query time:

**Offline:** `Load → Chunk → Embed → Store (FAISS)`  
**Query time:** `Retrieve top-k → Generate`

### 1.1 Chunk
Our docs are short, so one chunk per doc keeps the demo readable. On real data you'd split into ~200–500 token windows with overlap.

In [ ]:
def chunk_text(text, max_words=60, overlap=10):
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+max_words]))
        i += max_words - overlap
    return chunks

chunks, chunk_meta = [], []
for doc_id, text in DOCS.items():
    for j, c in enumerate(chunk_text(text)):
        chunks.append(c)
        chunk_meta.append({"doc": doc_id, "chunk": j})

print(f"{len(chunks)} chunks created")

### 1.2 Embed + store in FAISS
We L2-normalize embeddings (in `embed`) and use an **inner-product** index, so the score is cosine similarity.

In [ ]:
import faiss

chunk_vecs = embed(chunks)                      # (n, dim)
dim = chunk_vecs.shape[1]
index = faiss.IndexFlatIP(dim)                  # inner product on normalized vecs = cosine
index.add(chunk_vecs)
print(f"✅ FAISS index built: {index.ntotal} vectors, dim={dim}")

### 1.3 Retrieve + generate
Embed the query the same way, pull the top-k chunks, stuff them into the prompt, and ask the LLM.

In [ ]:
def rag_retrieve(query, k=3):
    qv = embed(query)
    scores, idx = index.search(qv, k)
    return [(chunks[i], chunk_meta[i], float(scores[0][r])) for r, i in enumerate(idx[0])]

def rag_answer(query, k=3, verbose=True):
    hits = rag_retrieve(query, k)
    context = "\n\n".join(f"[{m['doc']}] {c}" for c, m, _ in hits)
    prompt = (f"Answer the question using ONLY the context. "
              f"If the answer isn't fully supported, say so.\n\n"
              f"Context:\n{context}\n\nQuestion: {query}\nAnswer:")
    ans = llm(prompt)
    if verbose:
        print("🔎 Retrieved chunks:")
        for c, m, s in hits:
            print(f"   • {m['doc']}  (cos={s:.3f})")
        print("\n💬 Answer:\n" + ans)
    return ans, hits

_ = rag_answer("Who founded Orion Labs and what does it build?")

### 1.4 Now break it — a multi-hop question
Ask something whose answer is **spread across three documents**. Watch the retrieved chunks: similarity search grabs text that *looks* like the question, but the connecting facts may never make it into context.

In [ ]:
rag_multi, _ = rag_answer("How is Lena Ortiz connected to NimbusDB?")

> **Observe:** vector search ranks chunks by surface similarity to the query. The Ortiz→Vexel→Cumulo→NimbusDB chain lives in *different* chunks that aren't all similar to the question, so the answer is often incomplete or hedged. **This is the motivation for Graph RAG.**

---
## Part 2 · Graph RAG (from scratch)

Same corpus, different index. Instead of flat vectors we build a **knowledge graph**:

`Chunk → LLM extracts (entity, relation, entity) triples → networkx graph → traverse at query time`

This is the core idea behind Microsoft GraphRAG and LightRAG, stripped to its essentials.

### 2.1 Extract entities & relations with the LLM
We ask the model to return triples as JSON. This is the expensive, offline step (one LLM call per chunk) — and the reason graph indexing costs more than embedding.

In [ ]:
import json, re

EXTRACT_SYS = "You extract knowledge graphs. Return ONLY valid JSON, no prose."
EXTRACT_PROMPT = """From the text, extract entities and their relationships.
Return JSON: {{"triples": [{{"source": "...", "relation": "...", "target": "..."}}]}}
Use canonical entity names (e.g. people, companies, products). Keep relations short (e.g. "founded", "acquired", "builds", "led_by", "integrates_with").

Text:
{text}"""

def extract_triples(text):
    raw = llm(EXTRACT_PROMPT.format(text=text), system=EXTRACT_SYS)
    raw = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(raw).get("triples", [])
    except Exception:
        return []

all_triples = []
for doc_id, text in DOCS.items():
    t = extract_triples(text)
    for tr in t: tr["doc"] = doc_id
    all_triples += t
    print(f"{doc_id}: {len(t)} triples")

print(f"\nTotal: {len(all_triples)} triples")
for tr in all_triples[:8]:
    print(f"   ({tr['source']}) -[{tr['relation']}]-> ({tr['target']})")

### 2.2 Build the knowledge graph
Load the triples into a `networkx` graph. Nodes = entities, edges = relations (we keep the source doc on each edge for traceability).

In [ ]:
import networkx as nx

G = nx.DiGraph()
for tr in all_triples:
    s, r, t = tr["source"].strip(), tr["relation"].strip(), tr["target"].strip()
    if s and t:
        G.add_node(s); G.add_node(t)
        G.add_edge(s, t, relation=r, doc=tr["doc"])

print(f"✅ graph: {G.number_of_nodes()} entities, {G.number_of_edges()} relationships")
print("\nEntities:", ", ".join(sorted(G.nodes())[:20]))

### 2.3 Visualize (optional)
A quick look at the structure the graph captured.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))
pos = nx.spring_layout(G, seed=42, k=0.9)
nx.draw_networkx_nodes(G, pos, node_color="#0E9F6E", node_size=1500, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=8, font_color="white")
nx.draw_networkx_edges(G, pos, edge_color="#94a3b8", arrows=True, arrowsize=12)
nx.draw_networkx_edge_labels(G, pos, edge_labels={(u, v): d["relation"] for u, v, d in G.edges(data=True)}, font_size=6.5)
plt.axis("off"); plt.title("Knowledge graph extracted from the corpus"); plt.tight_layout(); plt.show()

### 2.4 Query by traversal
Strategy: **(1)** find the entities mentioned in the question, **(2)** walk their neighbourhood to collect connected facts, **(3)** hand that subgraph to the LLM as context. The traversal is what lets us answer multi-hop questions that flat retrieval misses.

In [ ]:
def find_seed_entities(query):
    """Match graph entities to the query (simple, robust substring + token match)."""
    q = query.lower()
    seeds = [n for n in G.nodes() if n.lower() in q]
    if not seeds:  # fall back to token overlap
        qtokens = set(re.findall(r"\w+", q))
        for n in G.nodes():
            if qtokens & set(re.findall(r"\w+", n.lower())):
                seeds.append(n)
    return list(dict.fromkeys(seeds))

def subgraph_context(seeds, hops=2):
    """Collect facts within `hops` of any seed entity (undirected walk)."""
    UG = G.to_undirected()
    keep = set(seeds)
    for s in seeds:
        if s in UG:
            keep |= set(nx.single_source_shortest_path_length(UG, s, cutoff=hops).keys())
    facts = []
    for u, v, d in G.edges(data=True):
        if u in keep and v in keep:
            facts.append(f"{u} -[{d['relation']}]-> {v}   (source: {d['doc']})")
    return facts

def graph_answer(query, hops=2, verbose=True):
    seeds = find_seed_entities(query)
    facts = subgraph_context(seeds, hops)
    context = "\n".join(facts) if facts else "(no connected facts found)"
    prompt = (f"Use the knowledge-graph facts to answer. Trace the connection step by step.\n\n"
              f"Facts:\n{context}\n\nQuestion: {query}\nAnswer:")
    ans = llm(prompt)
    if verbose:
        print(f"🔵 Seed entities: {seeds}")
        print(f"🕸️  Sub-graph facts ({len(facts)}):")
        for f in facts: print("   • " + f)
        print("\n💬 Answer:\n" + ans)
    return ans, facts

graph_multi, _ = graph_answer("How is Lena Ortiz connected to NimbusDB?")

> **Compare with Part 1.4.** The graph path assembles the full `Ortiz → Vexel → Cumulo → NimbusDB` chain and explains it — *with the source doc for every hop*. That traceability is a core Graph RAG advantage.

Graph RAG also enables **global questions** ("what are the main themes?") via community summaries — we keep the demo to traversal, but that map-reduce-over-communities step is what Microsoft GraphRAG adds on top.

---
## Part 3 · Hybrid — route each query to the right engine

Neither approach wins every query. A **router** classifies the query and dispatches:

- *Simple / factual / single-entity* → **Traditional RAG** (fast, cheap)
- *Multi-hop / relational / "how is X connected to Y"* → **Graph RAG**
- *Broad / ambiguous* → **both**, then let the LLM merge

We start with an LLM classifier (most flexible). A keyword/length heuristic is shown too — in production you often begin with the heuristic and graduate to a classifier.

In [ ]:
ROUTER_SYS = "You are a query router. Reply with ONE word only."
ROUTER_PROMPT = """Classify the query into exactly one route:
- "vector"  : a simple, factual, single-entity lookup
- "graph"   : needs multi-hop reasoning or relationships between entities
- "both"    : broad/ambiguous, or benefits from both

Query: {q}
Route:"""

def route(query):
    r = llm(ROUTER_PROMPT.format(q=query), system=ROUTER_SYS).lower()
    for tag in ("vector", "graph", "both"):
        if tag in r: return tag
    return "vector"

def hybrid_answer(query, verbose=True):
    r = route(query)
    print(f"🧭 Router → {r.upper()}  for: {query!r}\n")
    if r == "vector":
        ans, _ = rag_answer(query, verbose=verbose)
    elif r == "graph":
        ans, _ = graph_answer(query, verbose=verbose)
    else:
        v, _ = rag_answer(query, verbose=False)
        g, _ = graph_answer(query, verbose=False)
        ans = llm(f"Merge these two drafts into one complete, non-redundant answer.\n\n"
                  f"Draft A (vector): {v}\n\nDraft B (graph): {g}\n\nQuestion: {query}\nFinal answer:")
        if verbose: print("💬 Merged answer:\n" + ans)
    return ans

print("="*70)
hybrid_answer("Who is the CEO of Cumulo Cloud?")        # expect → vector
print("\n" + "="*70)
hybrid_answer("How is Lena Ortiz connected to NimbusDB?")  # expect → graph

---
## Part 4 · Compare all three, side by side

Run a fixed question set through each engine and eyeball the differences. Watch how the router's choice tracks the question type.

In [ ]:
QUESTIONS = [
    "Who founded Orion Labs?",                              # simple → vector is enough
    "How is Lena Ortiz connected to NimbusDB?",            # multi-hop → graph wins
    "Which products can be combined for a hybrid system?", # relational-ish
    "What happened to Vexel Analytics after 2022?",        # cross-doc
]

import textwrap
def short(s, n=300): return textwrap.shorten(s.replace("\n", " "), n)

for q in QUESTIONS:
    print("█"*78)
    print("Q:", q)
    v, _ = rag_answer(q, verbose=False)
    g, _ = graph_answer(q, verbose=False)
    chosen = route(q)
    print(f"\n  [VECTOR] {short(v)}")
    print(f"  [GRAPH ] {short(g)}")
    print(f"  [ROUTER] → {chosen.upper()}")
    print()

### What to take away
- **Simple, single-fact questions:** vector RAG matches the graph at a fraction of the cost.
- **Multi-hop / relational questions:** the graph answers completely and *traceably*; vector RAG hedges or misses links.
- **The router** gives you the graph's depth only when a query needs it — the essence of the hybrid pattern.

### A lightweight evaluation harness (optional)
For real projects, replace eyeballing with metrics. A minimal LLM-as-judge faithfulness check:

In [ ]:
def judge_faithfulness(question, answer, context):
    verdict = llm(
        f"Is the ANSWER fully supported by the CONTEXT? Reply 'yes' or 'no' then one short reason.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER: {answer}",
        system="You are a strict grader.")
    return verdict

q = "How is Lena Ortiz connected to NimbusDB?"
ans, facts = graph_answer(q, verbose=False)
print(judge_faithfulness(q, ans, "\n".join(facts)))

> In production, use a purpose-built evaluator such as **Ragas** (faithfulness, answer relevancy, context recall/precision) over a small **golden Q&A set**, and run it on every change.

---
## 5 · From notebook to production

What we hand-built maps directly onto mature libraries:

| We built (from scratch) | Production equivalent |
|--------------------------|------------------------|
| FAISS + `embed()` | FAISS / Chroma / Qdrant / Weaviate, via **LangChain** or **LlamaIndex** |
| LLM triple extraction + networkx | **Microsoft GraphRAG**, **LightRAG**, **nano-graphrag** |
| Graph store | **Neo4j** (`neo4j-graphrag`, `llm-graph-builder`) |
| LLM router | LangGraph / LlamaIndex router, or a small trained classifier |
| LLM-as-judge | **Ragas**, Open-RAG-Eval |

**Swapping the LLM / going local:** only the two helpers in **0.3** depend on the provider. Point `client` at any OpenAI-compatible endpoint (e.g. **Ollama**: `OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")`, `CHAT_MODEL="llama3.1"`), and use a local embedding model (e.g. `sentence-transformers`) in `embed()`. Everything else runs unchanged.

### 📚 References (high-star, current as of June 2026)
- **microsoft/graphrag** — canonical Graph RAG (entity graph + community summaries, local/global search). `https://github.com/microsoft/graphrag` (~33.6k⭐)
- **HKUDS/LightRAG** — simple & fast KG-RAG, dual-level retrieval, cheap incremental updates. `https://github.com/HKUDS/LightRAG` (~36.5k⭐)
- **gusye1234/nano-graphrag** — tiny, readable GraphRAG; great for learning internals. `https://github.com/gusye1234/nano-graphrag`
- **NirDiamant/RAG_Techniques** — 40+ notebook tutorials (simple, adaptive, corrective, agentic, graph). `https://github.com/NirDiamant/RAG_Techniques` (~27.8k⭐)
- **neo4j/neo4j-graphrag** + **neo4j-labs/llm-graph-builder** — KGs from unstructured data; production graph retrievers.
- **Awesome-GraphRAG** — curated surveys, papers, benchmarks, projects.
- **From Local to Global: A Graph RAG Approach** (Edge et al., 2024) — the GraphRAG paper. `https://arxiv.org/abs/2404.16130`
- **LightRAG** (Guo et al., 2024) — `https://arxiv.org/abs/2410.05779`

---
*Built for the AAA Learning Session. Reuse and adapt freely.*